# Auto Batch Size Tuning

When running parameter-shift gradients on a GPU, the number of shifted circuits evaluated in parallel (the **chunk size**) determines both speed and memory usage. Too large and you get an OOM error; too small and you leave GPU capacity on the table.

qiskit-trev provides `auto_batch_size` to automatically find the largest chunk size that fits in GPU memory. This tutorial shows:

1. How auto batch size works under the hood
2. Using it with `BatchParameterShiftGradient`
3. Using it with `QMLModel`
4. Calling `auto_batch_size` directly for custom workloads

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import torch
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit_trev import TensorRingModel, BatchParameterShiftGradient
from qiskit_trev.optimization.auto_batch import auto_batch_size

## 1. The Problem: Choosing Chunk Size

For a circuit with `P` parameters, the parameter-shift rule requires `2P` circuit evaluations (one +shift and one -shift per parameter). Evaluating all `2P` at once is fastest, but may exceed GPU memory for large circuits.

You can set `chunk_size` manually, but the right value depends on your GPU, circuit depth, qubit count, and bond dimension — hard to guess.

In [ ]:
# A circuit with many parameters — 6 qubits, 4 layers = 24 params
n_qubits = 6
n_layers = 4

qc = QuantumCircuit(n_qubits)
for layer in range(n_layers):
    for q in range(n_qubits):
        qc.ry(0.0, q)
    for q in range(n_qubits - 1):
        qc.cx(q, q + 1)

n_params = n_qubits * n_layers
print(f"Circuit: {n_qubits} qubits, {n_layers} layers, {n_params} parameters")
print(f"Parameter-shift requires {2 * n_params} circuit evaluations per gradient")

## 2. Auto Batch Size with `BatchParameterShiftGradient`

The easiest way to use auto batch sizing is to pass `chunk_size="auto"` when creating a `BatchParameterShiftGradient`. On the first gradient call, it probes the GPU to find the largest chunk that fits, then reuses that value for all subsequent calls.

In [ ]:
H = SparsePauliOp.from_list([
    ("ZZIIII", 1.0), ("IZZIII", 1.0), ("IIZZII", 1.0),
    ("IIIZZI", 1.0), ("IIIIZZ", 1.0),
])

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TensorRingModel(qc, H, rank=8, device=device)

# chunk_size="auto" — probes GPU memory on first call
grad_fn = BatchParameterShiftGradient(model, chunk_size="auto")

theta = torch.randn(n_params)
grad = grad_fn(theta)

print(f"Device: {device}")
print(f"Resolved chunk size: {grad_fn._resolved_chunk_size}")
print(f"Gradient shape: {grad.shape}")
print(f"Gradient: {grad[:4].tolist()}  ...")  # first 4 entries

## 3. Comparing Chunk Size Strategies

Let's compare the three options:
- `chunk_size=None` — evaluate all `2P` circuits at once (fast, but may OOM)
- `chunk_size=4` — fixed small chunks (safe, but slow)
- `chunk_size="auto"` — automatically tuned (best of both worlds)

In [ ]:
import time

theta = torch.randn(n_params)

strategies = {
    "None (all at once)": None,
    "Fixed (chunk_size=4)": 4,
    "Auto": "auto",
}

for name, cs in strategies.items():
    grad_fn = BatchParameterShiftGradient(model, chunk_size=cs)

    # Warmup
    _ = grad_fn(theta)

    # Time it
    start = time.perf_counter()
    for _ in range(5):
        grad = grad_fn(theta)
    elapsed = (time.perf_counter() - start) / 5

    resolved = grad_fn._resolved_chunk_size or "all"
    print(f"{name:25s}  chunk={str(resolved):>5s}  time={elapsed*1000:.1f} ms")

## 4. Using Auto Batch Size in Optimizers

`GradientOptimizer` accepts the same `chunk_size` parameter, which it passes through to `BatchParameterShiftGradient` internally.

In [ ]:
from qiskit_trev import GradientOptimizer

# Use chunk_size="auto" in the optimizer
grad_opt = GradientOptimizer(lr=0.1, optimizer_cls="adam", chunk_size="auto")

theta0 = torch.randn(n_params) * 0.1
result = grad_opt.minimize(model, theta0, max_iter=30)

print(f"Final energy: {result.cost:.4f}")
print(f"Iterations:   {result.num_iterations}")

## 5. Calling `auto_batch_size` Directly

For custom workloads, you can call `auto_batch_size` yourself. You provide a function that runs a trial batch of a given size, and it returns the largest size that fits in GPU memory.

The algorithm:
1. **Exponential growth** — doubles the batch size until OOM
2. **Binary search** — narrows down to the exact maximum between the last success and first failure

In [ ]:
from qiskit_trev.optimization.auto_batch import auto_batch_size
from qiskit_trev import TensorRingState

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define a trial function that simulates the actual workload
def trial_run(batch_size):
    """Simulate evaluating `batch_size` shifted circuits."""
    base = torch.zeros(1, n_params, device=device)
    batch = base.expand(2 * batch_size, -1).clone()
    state = TensorRingState(n_qubits, rank=8, device=str(device))
    state.build_batch(model._gate_templates, batch)

optimal_bs = auto_batch_size(
    trial_run,
    device,
    min_bs=1,
    max_bs=min(4096, n_params),
    safety_frac=0.85,  # use 85% of free memory
)
print(f"Optimal batch size: {optimal_bs}")

## 6. Parameters Reference

`auto_batch_size` accepts these keyword arguments:

| Parameter | Default | Description |
|---|---|---|
| `min_bs` | 1 | Minimum batch size (returned if even this fails) |
| `max_bs` | 65536 | Upper bound for the search |
| `safety_frac` | 0.85 | Fraction of free GPU memory to target |
| `growth` | 2.0 | Exponential growth factor during probing |
| `warmup` | 2 | Number of warmup runs per trial (for stable timing) |

**Tips:**
- Lower `safety_frac` (e.g. 0.7) if other processes share the GPU
- On CPU, `auto_batch_size` always returns `min_bs` — chunking is only useful for GPU memory management
- The result is cached after the first call in both `BatchParameterShiftGradient` and `QMLModel`, so the probing cost is paid only once